# Gemini API — Glass Ball Impact Video

Generate **20 frames** with Gemini native image generation, then stitch them into an MP4 showing a glass ball falling and hitting the ground.

**Requirements:** `GEMINI_API_KEY` in `~/Workspace/.env` or Colab Secrets.

In [ ]:
%pip install -q google-genai pillow imageio imageio-ffmpeg

In [ ]:
import os
from io import BytesIO
from pathlib import Path

import imageio.v2 as imageio
from google import genai
from google.genai import types
from IPython.display import Image as IPImage, Video, display
from PIL import Image

NOTEBOOK_DIR = Path.cwd()
FRAMES_DIR = NOTEBOOK_DIR / "frames_glass_ball"
VIDEO_PATH = NOTEBOOK_DIR / "glass_ball_impact.mp4"
NUM_FRAMES = 20
FPS = 8
IMAGE_MODEL = "gemini-2.0-flash-preview-image-generation"


def load_api_key() -> str:
    """Load GEMINI_API_KEY from Colab secrets or ~/Workspace/.env."""
    try:
        from google.colab import userdata
        return userdata.get("GEMINI_API_KEY")
    except ImportError:
        pass

    env_file = Path.home() / "Workspace" / ".env"
    if env_file.exists():
        for line in env_file.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        raise ValueError(
            "GEMINI_API_KEY not found. Add it to ~/Workspace/.env or Colab Secrets."
        )
    return key


client = genai.Client(api_key=load_api_key())
print("Gemini client ready.")

In [ ]:
BASE_STYLE = (
    "Photorealistic 3D render, side profile view, fixed camera angle, "
    "soft studio lighting, dark neutral background, gray concrete ground plane. "
    "Subject: one clear glass sphere (glass ball) with realistic reflections and refraction. "
    "Keep composition, lighting, and camera identical across frames; only change the motion state. "
)

FRAME_STAGES = [
    "The glass ball is stationary at the very top of the frame, far above the ground.",
    "The glass ball has just begun to fall; a tiny gap below it, subtle downward motion.",
    "The glass ball is falling in the upper quarter of the frame.",
    "The glass ball is falling in the upper third of the frame.",
    "The glass ball is falling in the upper-middle area of the frame.",
    "The glass ball is falling through the middle of the frame.",
    "The glass ball is falling in the lower-middle area of the frame.",
    "The glass ball is falling in the lower third of the frame.",
    "The glass ball is falling in the lower quarter of the frame.",
    "The glass ball is very close to the ground, about to touch.",
    "The glass ball is millimeters above the ground, maximum tension before impact.",
    "First contact: the glass ball touches the ground and slightly flattens at the bottom.",
    "Impact: the ball compresses on the ground, hairline cracks appear on the surface.",
    "Impact peak: cracks spread across the glass ball, small shards begin to separate.",
    "The glass ball shatters; large curved glass fragments fly outward.",
    "More fragments scatter across the ground, the sphere is breaking apart.",
    "Glass shards spread on the concrete floor, motion blur on fast pieces.",
    "Fragments are slowing down and settling on the ground.",
    "Most shards rest on the floor; a few pieces still tumbling.",
    "Final frame: scattered glass shards on the ground, no intact sphere, dust settling.",
]

FRAME_PROMPTS = [
    f"{BASE_STYLE} Frame {i + 1} of {NUM_FRAMES}: {stage}"
    for i, stage in enumerate(FRAME_STAGES)
]

print(f"Prepared {len(FRAME_PROMPTS)} frame prompts.")

In [ ]:
def extract_image_from_response(response) -> Image.Image:
    """Return the first inline image from a Gemini response."""
    if not response.candidates:
        raise RuntimeError("No candidates in response")
    for part in response.candidates[0].content.parts:
        if part.inline_data and part.inline_data.data:
            return Image.open(BytesIO(part.inline_data.data))
    raise RuntimeError("No image returned — try re-running this frame.")


def generate_frame(prompt: str, frame_index: int) -> Image.Image:
    """Generate a single frame with Gemini image generation."""
    response = client.models.generate_content(
        model=IMAGE_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_modalities=["IMAGE"],
        ),
    )
    img = extract_image_from_response(response)
    out_path = FRAMES_DIR / f"frame_{frame_index:02d}.png"
    img.save(out_path)
    print(f"Saved {out_path.name} ({img.size[0]}x{img.size[1]})")
    return img


FRAMES_DIR.mkdir(parents=True, exist_ok=True)
frames: list[Image.Image] = []

for i, prompt in enumerate(FRAME_PROMPTS, start=1):
    frame_path = FRAMES_DIR / f"frame_{i:02d}.png"
    if frame_path.exists():
        img = Image.open(frame_path)
        print(f"Loaded cached {frame_path.name}")
    else:
        print(f"Generating frame {i}/{NUM_FRAMES}...")
        img = generate_frame(prompt, i)
    frames.append(img.convert("RGB"))

print(f"\nAll {len(frames)} frames ready in {FRAMES_DIR}")

In [ ]:
import numpy as np

# Resize frames to a common size for smooth video encoding
target_size = frames[0].size
video_frames = [
    np.array(frame.resize(target_size, Image.Resampling.LANCZOS))
    for frame in frames
]

imageio.mimsave(VIDEO_PATH, video_frames, fps=FPS, codec="libx264", quality=8)
print(f"Video saved: {VIDEO_PATH} ({len(video_frames)} frames @ {FPS} fps)")

display(Video(str(VIDEO_PATH), embed=True, width=640))

In [ ]:
# Preview a few key frames inline
preview_indices = [0, 4, 9, 12, 15, 19]
for idx in preview_indices:
    path = FRAMES_DIR / f"frame_{idx + 1:02d}.png"
    if path.exists():
        print(f"Frame {idx + 1}")
        display(IPImage(filename=str(path), width=320))